In [ ]:
!pip install -U transformers
!pip install langid

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM, AutoTokenizer, pipeline
from transformers import BitsAndBytesConfig
import torch
import pickle
import random
from sentence_transformers import util
from datasets import load_dataset
import langid

Prepare XSum dataset

In [ ]:
xsum = load_dataset("EdinburghNLP/xsum", revision="refs/convert/parquet")
testdocs = xsum["test"]["document"][200:]
valdocs = xsum["validation"]["document"][100:]

In [ ]:
# checks if the document is in english
def is_english(text):
  lang, score = langid.classify(text.replace("\n", " ")[:500])
  return lang == "en"

**Change to false for test set calculation:**

In [ ]:
use_validation = False

In [ ]:
docs = valdocs if use_validation else testdocs

en_docs = [doc for doc in docs if is_english(doc)]

## Get DistilBART outputs and MExGen explanations for Prompt Examples

**Change to false for similarity based examples:**

In [ ]:
use_cluster_examples = False

In [ ]:
# load the closest indices file

if use_validation:
  if use_cluster_examples:
    with open("closest_indices_val_cluster.pkl", "rb") as f:
      closest_indices = pickle.load(f)
  else:
    with open("closest_indices_val_sim.pkl", "rb") as f:
      closest_indices = pickle.load(f)
else:
  if use_cluster_examples:
    with open("closest_indices_test_cluster.pkl", "rb") as f:
      closest_indices = pickle.load(f)
  else:
    with open("closest_indices_test_sim.pkl", "rb") as f:
      closest_indices = pickle.load(f)

This was taken from the MExGen repository \url…

In [ ]:
# Clone the repository after removing the old copy if any
!rm -rf ICX360
!git clone https://github.com/IBM/ICX360.git

# Install the package
%cd ICX360
!pip install uv
!uv pip install .
%cd ..

In [ ]:
# import standard packages
import warnings
warnings.filterwarnings("ignore")

from IPython.display import display
import pandas as pd
import torch
from transformers import BartForConditionalGeneration, BartTokenizerFast

# ICX360 classes and functions
from icx360.algorithms.mexgen import CLIME    # explainer
from icx360.utils.coloring_utils import color_units    # highlight and display text
from icx360.utils.general_utils import select_device    # set device automatically
from icx360.utils.model_wrappers import HFModel    # model wrapper

In [ ]:
device = select_device()
display(device)

In [ ]:
model_name = "sshleifer/distilbart-xsum-12-6"
model = BartForConditionalGeneration.from_pretrained(model_name).to(device)
tokenizer = BartTokenizerFast.from_pretrained(model_name, add_prefix_space=True)

In [ ]:
# wrap the model with a common API (HFModel) that the explainer will use
wrapped_model = HFModel(model, tokenizer)

In [ ]:
# attributes an importance score to each part of the input document by calling the summarization model on perturbed versions of the input.
explainer = CLIME(wrapped_model, scalarizer="prob", segmenter="en_core_web_sm", device=device)

End of copy from MexGen code

In [ ]:
def mixed_level_attributions(documents, closest_indices, explainer, n_over_d=10, K=3, segment_type="s", k=3, phi=1/3):

  all_results = []
  instance_count = 0

  for sublist in closest_indices:
    print(instance_count)

    for index in sublist:
      print(f"index: {index}")
      doc = documents[index]

      # ---- Sentence-level attributions ----
      # (1) - compute attribution scores
      output_dict_sent = explainer.explain_instance(
          doc,
          oversampling_factor=n_over_d,
          max_units_replace=K,
          segment_type=segment_type
          )
      # sentences (units)
      sent_units = output_dict_sent["attributions"]["units"]
      # raw sentence attribution scores - ξ
      xi = np.array(output_dict_sent["attributions"]["prob"], dtype=float)

      # (2) - normalize scores ψ in [-1,1]
      xi_min, xi_max = np.min(xi), np.max(xi)
      if xi_max == xi_min:
        psi = np.zeros_like(xi)
      else:
        psi = 2 * ( (xi - xi_min) / (xi_max - xi_min) ) - 1

      # (3) - compute Top-k ψ - k largest ψ scores
      top_k_idx = np.argsort(psi)[::-1][:k]
      top_k_mask = np.zeros_like(psi, dtype=bool)
      top_k_mask[top_k_idx] = True

      # (4) - (7) - score is higher/equal to threshold φ and in top-k
      refine_mask = (psi >= phi) & top_k_mask
      units_to_be_refined = np.where(refine_mask)[0].tolist()

      # ---- Phrase-level attributions for chosen sentences----
      phrase_explanations = {}
      for idx in units_to_be_refined:
        sentence = sent_units[idx]
        output_dict_phrase = explainer.explain_instance(
            sentence,
            oversampling_factor=n_over_d,
            max_units_replace=K,
            segment_type="ph"
            )
        attributions = output_dict_phrase["attributions"]
        # save attribution dictionary for every top sentence
        phrase_explanations[idx] = attributions

      instance = {
          "inst_id": instance_count,
          "doc_id": int(index),
          "doc": doc,
          "distilbart_sum": output_dict_sent["output_orig"].output_text[0],
          "sentence_attributions": output_dict_sent["attributions"],
          "refined_indices": units_to_be_refined,
          "phrase_attributions": phrase_explanations
          }
      all_results.append(instance)

    instance_count += 1

  return all_results


**Calculating the explanations for the 300 validation set examples takes ~22 minutes with the L4 GPU of Google Colab!** <br>
**Calculating the explanations for the 600 test set examples takes ~42 minutes with the L4 GPU of Google Colab!**

toma_get_probs log: <br>
That’s the function ICX360 calls internally to get the scalarizer values for each perturbed input. To save time, the perturbations are batched together instead of running them one by one.

In [ ]:
all_results = mixed_level_attributions(en_docs, closest_indices, explainer)

# Get highest scored Sentences and Phrases

### 1. HIGHEST SCORED SENTENCE

In [ ]:
for exp in all_results:
  explanation = exp["sentence_attributions"]
  probs = np.array(explanation["prob"])
  sentences = explanation["units"]

  # Find index of the highest score
  max_idx = np.argmax(probs)
  top_score = float(probs[max_idx])
  top_sentence = sentences[max_idx].strip()


  # Save back into dict
  exp["top_sentence"] = {"score": top_score, "sentence": top_sentence}

### 2. TOP 2 HIGHEST SCORED SENTENCES

In [ ]:
for exp in all_results:
    explanation = exp["sentence_attributions"]
    probs = np.array(explanation["prob"])
    sentences = explanation["units"]

    # Get indices of top-2 scores (sorted highest → lowest)
    top2_idx = np.argsort(probs)[::-1][:2]

    top_sentences = []
    for idx in top2_idx:
        top_sentences.append({
            "score": float(probs[idx]),
            "sentence": sentences[idx].strip()
        })

    exp["top_2_sentences"] = top_sentences


### 3. HIGHEST SCORED PHRASE (per refined sentence) - Multi-Level

In [ ]:
for exp in all_results:
    # all refined sentences
    phrase_attr_all = exp["phrase_attributions"]
    top_phrases_all = {}

    # loop over each refined sentence that has phrase-level attribution
    for sent_idx, phrase_attr in phrase_attr_all.items():
        probs_ph = np.array(phrase_attr["prob"])
        phrases = phrase_attr["units"]

        # find the indices of the highest scores
        max_idx = np.argmax(probs_ph)
        top_score = float(probs_ph[max_idx])
        top_phrase = phrases[max_idx].strip()

        # save best phrase per sentence index
        top_phrases_all[int(sent_idx)] = {
            "score": top_score,
            "phrase": top_phrase
        }

    # save back into dict
    exp["top_phrases"] = top_phrases_all

### 4. TOP HIGHEST SCORED PHRASES (per refined sentence) - Multi-Level

In [ ]:
for exp in all_results:
    phrase_attr_all = exp["phrase_attributions"]
    top_phrases_all = {}

    for sent_idx, phrase_attr in phrase_attr_all.items():
        probs_ph = np.array(phrase_attr["prob"])
        phrases = phrase_attr["units"]

        # sort indices by score
        sorted_idx = np.argsort(probs_ph)   # ascending
        top_two_idx = sorted_idx[-2:]       # last two are highest

        # build list of top two (sorted highest → second-highest)
        top_two = []
        for idx in reversed(top_two_idx):
            top_two.append({
                "score": float(probs_ph[idx]),
                "phrase": phrases[idx].strip()
            })

        top_phrases_all[int(sent_idx)] = top_two

    exp["top_2_phrases"] = top_phrases_all

In [ ]:
if use_validation:
  if use_cluster_examples:
    with open("EXAMPLES_300_VAL_CLUSTER.pkl", "wb") as f:
      pickle.dump(all_results, f)
    with open("EXAMPLES_300_VAL_CLUSTER.pkl", "rb") as f:
      examples = pickle.load(f)
  else:
    with open("EXAMPLES_300_VAL_SIM.pkl", "wb") as f:
      pickle.dump(all_results, f)
    with open("EXAMPLES_300_VAL_SIM.pkl", "rb") as f:
      examples = pickle.load(f)
else:
  if use_cluster_examples:
    with open("EXAMPLES_600_TEST_CLUSTER.pkl", "wb") as f:
      pickle.dump(all_results, f)
    with open("EXAMPLES_600_TEST_CLUSTER.pkl", "rb") as f:
      examples = pickle.load(f)
  else:
    with open("EXAMPLES_600_TEST_SIM.pkl", "wb") as f:
      pickle.dump(all_results, f)
    with open("EXAMPLES_600_TEST_SIM.pkl", "rb") as f:
      examples = pickle.load(f)

In [ ]:
#examples[0]